# Vectores y Embeddings

La detección de objetos nos dice *qué* hay en una imagen y *dónde* está. Pero algunas preguntas requieren más:
- ¿Es esta cara la misma persona que vimos antes?
- ¿Qué imagen almacenada es más similar a este nuevo recorte?
- ¿Puedo buscar por similitud semántica en lugar de coincidencia exacta de píxeles?

Responder estas preguntas requiere una representación diferente. En lugar de recuadros delimitadores, necesitamos una forma de representar el *significado* de una imagen como un número — o más precisamente, como una lista de números llamada **vector**. Este notebook desarrolla esa idea desde cero: desde la aritmética básica de vectores, pasando por las métricas de distancia, hasta los embeddings de CLIP en imágenes reales.

Al final también veremos cómo la misma idea de embedding se aplica al texto — lo que abre la puerta a la búsqueda multimodal y a los modelos de lenguaje.

## ¿Qué es un Vector?

Un **vector** es una lista ordenada de números. En NumPy es simplemente un array de 1 dimensión.
Geométricamente representa un punto — o una flecha desde el origen — en un espacio N-dimensional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Un vector de 2 dimensiones
v = np.array([3.0, 2.0])
print(f'Vector: {v}   Forma: {v.shape}   Tipo: {v.dtype}')

# Un vector de 512 dimensiones — mismo concepto, solo que no es directamente visualizable
big_v = np.random.randn(512)
print(f'Forma del vector de alta dimensión: {big_v.shape}')

In [ ]:
# Visualizar vectores 2D como flechas desde el origen
vectors = {
    'v = [3, 2]' : np.array([3, 2]),
    'w = [1, 4]' : np.array([1, 4]),
    '-v = [-3,-2]': np.array([-3, -2]),
}
fig, ax = plt.subplots(figsize=(5, 5))
colors = ['steelblue', 'darkorange', 'seagreen']
for (label, vec), color in zip(vectors.items(), colors):
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(vec[0] + 0.1, vec[1] + 0.1, label, color=color, fontsize=10)
ax.axhline(0, color='grey', lw=0.5)
ax.axvline(0, color='grey', lw=0.5)
ax.set_xlim(-4, 5); ax.set_ylim(-3, 5)
ax.set_aspect('equal')
ax.set_title('Vectores como flechas en el espacio 2D')
plt.tight_layout()
plt.show()

## Métricas de Distancia

Dados dos vectores **a** y **b**, podemos medir qué tan separados están.
Dos métricas comunes:

| Métrica | Fórmula | Cuándo preferirla |
|--------|---------|----------------|
| **Euclidiana** | $\\\|a - b\\\|_2 = \\\sqrt{\\\sum_i (a_i - b_i)^2}$ | Cuando la magnitud absoluta importa |
| **Similitud coseno** | $\\\frac{a \\\cdot b}{\\\|a\\\| \\\cdot \\\|b\\\|}$ | Cuando solo importa la dirección (ángulo) |

La **similitud coseno** ignora la longitud del vector — dos vectores idénticos escalados por cantidades diferentes obtienen similitud = 1.
Esto importa para los embeddings: una imagen más brillante debería seguir reconociéndose como la misma cara.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([1.0, 2.1, 2.9])  # ligeramente diferente
c = 5 * a                       # misma dirección que a, 5× más largo

def euclidean(x, y):
    return np.sqrt(np.sum((x - y) ** 2))

def cosine_similarity(x, y):
    # Producto escalar dividido por el producto de sus normas L2.
    # Rango: -1 (opuesto) a 1 (dirección idéntica).
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print(f'a vs b  — euclidiana: {euclidean(a, b):.3f}  coseno: {cosine_similarity(a, b):.4f}')
print(f'a vs c  — euclidiana: {euclidean(a, c):.3f}  coseno: {cosine_similarity(a, c):.4f}')
print()
print('a y c tienen la misma dirección → similitud coseno = 1.0 sin importar la escala')

In [ ]:
# Demostración visual: el coseno mide el ángulo, la euclidiana mide la distancia en línea recta
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
points = {'a': np.array([3, 1]), 'b': np.array([1, 3]), '2a': np.array([6, 2])}
colors = {'a': 'steelblue', 'b': 'darkorange', '2a': 'steelblue'}
styles = {'a': '-', 'b': '-', '2a': '--'}
for label, vec in points.items():
    for ax in axes:
        ax.annotate('', xy=vec, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=colors[label],
                                   lw=2, linestyle=styles[label]))
        ax.text(vec[0]+0.1, vec[1]+0.1, label, color=colors[label], fontsize=11)
a2, b2, two_a = points['a'], points['b'], points['2a']
# Euclidiana: distancia entre puntas
axes[0].plot([a2[0], b2[0]], [a2[1], b2[1]], 'r--', lw=1.5, label=f'euclid(a,b)={euclidean(a2,b2):.2f}')
axes[0].plot([a2[0], two_a[0]], [a2[1], two_a[1]], 'm--', lw=1.5, label=f'euclid(a,2a)={euclidean(a2,two_a):.2f}')
axes[0].set_title('Euclidiana: mide la distancia entre puntas')
axes[0].legend(fontsize=8)
# Coseno: mide el ángulo
angle_ab = np.degrees(np.arccos(np.clip(cosine_similarity(a2, b2), -1, 1)))
angle_a2a = np.degrees(np.arccos(np.clip(cosine_similarity(a2, two_a), -1, 1)))
axes[1].set_title('Coseno: mide el ángulo')
axes[1].text(4, 0.5, f'ángulo(a,b) = {angle_ab:.1f}°', fontsize=10, color='red')
axes[1].text(2, 2.5, f'ángulo(a,2a) = {angle_a2a:.1f}°', fontsize=10, color='purple')
for ax in axes:
    ax.set_xlim(-0.5, 7.5); ax.set_ylim(-0.5, 4)
    ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## ¿Qué son los Embeddings?

Un **embedding** es un mapeo aprendido de una entrada de alta dimensión (imagen, texto, audio)
a un vector denso de baja dimensión en un **espacio semántico**.

La propiedad clave: las entradas que son semánticamente similares se mapean a vectores que están **cerca**
en ese espacio — medido por la similitud coseno o la distancia euclidiana.

```
Imagen de cara A → [0.12, -0.34, 0.87, ...]  (512 números)
Imagen de cara A (iluminación diferente) → [0.11, -0.33, 0.85, ...]  ← ¡muy cerca!
Imagen de un coche  → [-0.55,  0.21, 0.03, ...]  ← lejos
```

Por eso los embeddings se usan para el reconocimiento facial: en lugar de comparar píxeles
(que cambian drásticamente con la iluminación/ángulo), comparamos **representaciones semánticas**.

## Embeddings de Imagen con CLIP

**CLIP** (Contrastive Language-Image Pre-training) se entrena en un conjunto muy grande de pares imagen-texto. Su idea clave es que no solo las imágenes se convierten en vectores, sino que **las imágenes y el texto se mapean al mismo espacio semántico**.

Para este módulo, usamos principalmente el lado de imagen de CLIP:
`imagen -> CLIP -> vector de 512 dimensiones`

Eso es suficiente para el reconocimiento facial y la búsqueda de similitud de imágenes.
Pero también apunta hacia una conexión posterior con NLP:
- una imagen puede embeberse,
- una descripción de texto también puede embeberse,
- y ambas pueden compararse en un espacio vectorial compartido.

Usamos `sentence-transformers` como un envoltorio conveniente alrededor del codificador CLIP.

### Cómo Funciona CLIP

CLIP (Contrastive Language-Image Pre-training) fue entrenado por OpenAI en 400 millones de pares imagen-texto extraídos de la web. El objetivo de entrenamiento es **contrastivo**: dado un lote de N pares (imagen, texto), el modelo se entrena para que los N pares coincidentes tengan alta similitud coseno y los N² - N pares no coincidentes tengan baja similitud.

La arquitectura tiene dos codificadores separados:
```
imagen  ──►  Vision Transformer (ViT-B/32)  ──►  vector de 512 dim
                                                    ↑ mismo espacio
texto   ──►  Transformer de texto             ──►  vector de 512 dim
```

`clip-ViT-B-32` significa: el backbone visual es un **ViT** (Vision Transformer), variante **B** (Base, ~86 M parámetros), tamaño de parche **32×32** píxeles.

Una propiedad clave de los vectores de salida: están **normalizados en L2** (norma unitaria). Esto significa que el producto escalar y la similitud coseno son idénticos (ya que ambas normas son 1, el denominador de la fórmula del coseno es 1). Por lo tanto, puedes comparar los embeddings de CLIP con un simple producto escalar, lo que hace las búsquedas de similitud muy eficientes.

Aquí usamos solo el codificador de imágenes. El lado del texto se vuelve importante cuando discutimos la búsqueda multimodal al final de este notebook.

In [ ]:
from PIL import Image
import os
from sentence_transformers import SentenceTransformer

# Cargar el codificador de imágenes CLIP (descarga ~350 MB en la primera ejecución)
clip_model = SentenceTransformer('clip-ViT-B-32')
print('Modelo cargado')
print(f'Dimensión del embedding de salida: {clip_model.get_sentence_embedding_dimension()}')

In [ ]:
# Usar imágenes de prueba locales del directorio resources/images
paths = {
    'lena': 'resources/images/lenna.png',
    'baboon': 'resources/images/baboon.png',
    'pepper': 'resources/images/peppers.jpg',
}
for name, path in paths.items():
    print(f'Listo: {path}')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4))
for ax, (name, path) in zip(axes, paths.items()):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Codificar cada imagen en un vector de 512 dimensiones
embeddings = {}
for name, path in paths.items():
    img = Image.open(path).convert('RGB')
    # encode() acepta una imagen PIL o lista de imágenes PIL
    emb = clip_model.encode(img)    # devuelve un array numpy
    embeddings[name] = emb
    norm = np.linalg.norm(emb)      # norma L2
    print(f'{name:10s}: shape={emb.shape}  norm={norm:.3f}  min={emb.min():.3f}  max={emb.max():.3f}')

In [ ]:
# Calcular similitudes coseno por pares
names = list(embeddings.keys())
n = len(names)
sim_matrix = np.zeros((n, n))
for i, ni in enumerate(names):
    for j, nj in enumerate(names):
        sim_matrix[i, j] = cosine_similarity(embeddings[ni], embeddings[nj])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim_matrix, vmin=0.7, vmax=1.0, cmap='viridis')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(names); ax.set_yticklabels(names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.3f}', ha='center', va='center', color='white', fontsize=10)
plt.colorbar(im, ax=ax, label='Similitud coseno')
ax.set_title('Similitud CLIP por pares entre imágenes de prueba')
plt.tight_layout()
plt.show()

## Visualizando el Espacio de Embeddings con PCA

512 dimensiones son imposibles de visualizar directamente.
**PCA** (Análisis de Componentes Principales) proyecta datos de alta dimensión a 2D preservando
tanta varianza como sea posible — útil para verificar rápidamente si imágenes similares se agrupan.

In [ ]:
from sklearn.decomposition import PCA

# Apilar todos los embeddings en una matriz: forma (n_imagenes, 512)
names_list = list(embeddings.keys())
X = np.stack([embeddings[n] for n in names_list])  # (3, 512)

# Proyectar a 2D
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)   # (3, 2)
explained = pca.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada por 2 componentes principales: {explained:.1f}%')

fig, ax = plt.subplots(figsize=(5, 5))
for i, name in enumerate(names_list):
    ax.scatter(*X_2d[i], s=150, zorder=3)
    ax.annotate(name, X_2d[i], textcoords='offset points', xytext=(6, 4), fontsize=11)
ax.set_title(f'Embeddings CLIP proyectados a 2D (PCA, {explained:.0f}% varianza)')
ax.set_xlabel('CP 1'); ax.set_ylabel('CP 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ¿Por qué Usar Vectores para el Reconocimiento Facial?

| Enfoque | Almacenamiento | Comparación | Escalabilidad |
|----------|---------|------------|-------|
| Píxeles crudos (512×512 RGB) | ~786 KB por cara | píxel a píxel, lento | deficiente |
| Embedding (512 flotantes) | 2 KB por cara | un solo producto escalar | excelente |

Una similitud coseno entre dos vectores de 512 dimensiones es solo un producto escalar y dos normas: una cantidad mínima de aritmética. Comparar píxeles crudos requiere cientos de miles de comparaciones directas.

Más importante aún, los embeddings son **robustos**: la misma persona bajo diferente iluminación, ángulo u oclusión parcial debería mapear a vectores cercanos. La comparación de píxeles generalmente fallaría.

Esta es también la conexión conceptual con el NLP. En los modelos de lenguaje posteriores, las frases y documentos se embeben por la misma razón: no porque las entradas crudas parezcan similares, sino porque significan cosas similares.

El siguiente notebook muestra cómo almacenar estos vectores y recuperar los más cercanos de manera eficiente con una base de datos vectorial.

## Embeddings en NLP: La Misma Idea para el Texto

Todo lo que hicimos arriba — codificar un objeto en un vector denso, medir la similitud con la distancia coseno, almacenar y buscar en una base de datos — se aplica igualmente al texto.

En NLP, un **modelo de lenguaje** mapea una frase o párrafo a un vector denso en un espacio semántico. Las frases que significan lo mismo terminan juntas; las frases no relacionadas terminan lejos. La práctica de buscar en un corpus calculando similitudes contra un vector de consulta se llama **búsqueda semántica**, y es mucho más poderosa que la coincidencia de palabras clave.

La siguiente celda de código demuestra esto directamente usando el lado de texto del modelo CLIP que ya está cargado.

In [ ]:
# CLIP mapea texto E imágenes al mismo espacio de 512 dimensiones.
# Por tanto, podemos comparar una consulta de texto directamente contra embeddings de imagen.
text_queries = [
    'a colourful image',
    'a human face',
    'a fruit or vegetable',
]
text_embeddings = clip_model.encode(text_queries)  # (3, 512)

print('Similitud coseno entre consultas de texto e imágenes:')
print(f'{"":15} ' + ' '.join(f'{q[:12]:>14}' for q in text_queries))
for img_name, img_emb in embeddings.items():
    sims = [cosine_similarity(img_emb, text_embeddings[i]) for i in range(len(text_queries))]
    print(f'{img_name:15} ' + ' '.join(f'{s:14.3f}' for s in sims))

El hecho de que los embeddings de texto e imagen vivan en el mismo espacio es lo que hace posible la **búsqueda multimodal**: puedes consultar una base de datos de imágenes con una descripción de texto, o encontrar documentos relacionados con una foto.

### Embeddings de Palabras y Modelos Estáticos

Antes de los grandes modelos de lenguaje, **word2vec** (Mikolov et al. 2013) y **GloVe** mostraron que las palabras individuales podían mapearse a vectores densos con sorprendentes propiedades geométricas:
```
vector('king') - vector('man') + vector('woman')  ≈  vector('queen')
```
Esta estructura de analogía — hoy considerada una simplificación, pero históricamente muy influyente — insinuaba que la geometría de un espacio semántico codifica relaciones del mundo real.

### Embeddings Contextuales y Transformers

word2vec produce el mismo vector para una palabra independientemente del contexto. BERT, GPT y sus sucesores introdujeron **embeddings contextuales**: el mismo token obtiene un vector diferente dependiendo de la frase en la que aparece. Esto es esencial para resolver ambigüedades ('banco' como institución financiera vs ribera de río).

### RAG: Generación Aumentada por Recuperación

La conexión entre embeddings y modelos de lenguaje alcanza su pico práctico actual en **RAG (Retrieval-Augmented Generation)**:
1. Un corpus de documentos se codifica en vectores y se almacena en una base de datos vectorial (como ChromaDB).
2. Cuando un usuario envía una consulta, esta se codifica en un vector.
3. Los documentos más similares se recuperan de la base de datos.
4. Esos documentos se inyectan en el contexto de un gran modelo de lenguaje.
5. El LLM genera una respuesta fundamentada que cita el contenido recuperado.

El paso de recuperación es exactamente lo que se demuestra en el notebook de ChromaDB — solo con imágenes y recortes de caras en lugar de documentos de texto.

### La Idea Central

Ya sea que la entrada sea un parche de imagen, un recorte de cara, una palabra o una frase:
> **Las cosas significativas que son semánticamente similares deben estar cerca en el espacio vectorial**.

Toda la maquinaria en este módulo — codificaciones CLIP, similitud coseno, búsquedas en ChromaDB — es una aplicación directa de esta única idea.